# ⭐ Aula 16 — Teorema do Limite Central e bons estimadores nas avaliações de um app

Este notebook **não tem nota nem entrega**, e você **não precisa saber programar**. É só executar as células, na ordem, de cima para baixo, e ler os textos entre elas.

Neste notebook você faz parte do **time de dados de uma loja de aplicativos**. Os usuários dão notas de 1 a 5 estrelas aos apps, e o time usa amostras dessas notas para tomar decisões. A Aula 16 traz duas ferramentas: o **Teorema do Limite Central (TLC)**, que diz como a média amostral se comporta, e as **qualidades de um bom estimador** (consistência, ausência de vício e eficiência).

> ⚠️ Os números usados aqui são **fictícios**, escolhidos só para ilustrar os modelos.

Cada seção segue sempre a mesma ordem:
1. **A situação e a pergunta** (o enunciado, como num exercício);
2. **A ferramenta da aula** que responde a pergunta;
3. **A resposta pela conta**;
4. **O que o código faz**, antes de cada célula;
5. **Como ler o resultado**.

> Notebook elaborado com auxílio de IA (Claude) e conferido pelo professor.

## Como executar

- Cada **célula de código** (as caixas cinza) tem um botão **▶** do lado esquerdo. Clique nele, ou clique na célula e aperte **Shift + Enter**.
- Execute **uma de cada vez, na ordem**: algumas células usam o resultado das anteriores.
- Na primeira vez, o Colab pode avisar que o notebook "não foi criado pelo Google". Clique em **Executar mesmo assim**.
- Se preferir, use o menu **Ambiente de execução → Executar tudo** e depois só leia, de cima para baixo.
- Detalhe: o Python escreve números com **ponto** no lugar da vírgula (`0.26` = 0,26).

## 0. Preparando as ferramentas

A célula abaixo carrega:

- **`numpy`**: uma "calculadora" que faz contas com milhares de números de uma vez e sabe **sortear** números aleatórios;
- **`matplotlib`**: desenha **gráficos**;
- **`scipy.stats.norm`**: a "tabela" da normal dentro do computador.

Criamos também o **sorteador** (`rng`) com a **semente 42**, que faz as simulações saírem sempre iguais.

**Lembrete:** **empírico** = o que sai da simulação; **teórico** = o que a fórmula da aula diz.

In [ ]:
import numpy as np                    # calculadora de números e sorteios
import matplotlib.pyplot as plt       # gráficos
from scipy.stats import norm          # a "tabela" da normal

rng = np.random.default_rng(42)       # sorteador, semente 42
print("Tudo pronto!")

## 1. O TLC numa população nada normal

### A situação
As notas de um app famoso seguem um padrão comum em lojas de aplicativos: quem gosta dá **5 estrelas**, quem teve problema dá **1 estrela**, e quase ninguém fica no meio. A distribuição das notas é:

| nota | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| probabilidade | 0,25 | 0,05 | 0,05 | 0,15 | 0,50 |

Com isso, $\mu = 3{,}6$ estrelas e $\sigma^2 = 2{,}84$ (desvio padrão $\sigma \approx 1{,}685$). Essa população tem formato de "U": nada parecido com um sino.

O time de produto acompanha a **média das últimas 40 avaliações** e se preocupa quando ela cai **abaixo de 3,2**.

### A pergunta
Se nada mudou no app, qual a chance de a média das últimas 40 notas ficar abaixo de 3,2 só por acaso?

### A ferramenta da aula
Pelo **TLC**, para $n$ grande, a média amostral é aproximadamente normal, **qualquer que seja** o formato da população:
$$\bar X \approx N\!\left(\mu,\ \frac{\sigma^2}{n}\right) .$$

### A resposta pela conta
$\sigma_{\bar X} = \dfrac{1{,}685}{\sqrt{40}} \approx 0{,}266$, então
$$P(\bar X < 3{,}2) \approx P\!\left(Z < \frac{3{,}2 - 3{,}6}{0{,}266}\right) = P(Z < -1{,}50) \approx 0{,}067 .$$

### O que o código faz
1. calcula $\mu$ e $\sigma$ a partir da tabela das notas;
2. sorteia **100.000 grupos de 40 avaliações** (`rng.choice` sorteia notas com as probabilidades da tabela) e calcula a média de cada grupo;
3. desenha, lado a lado, a distribuição das **notas individuais** e a das **médias de 40 notas**, com a curva normal do TLC por cima;
4. compara a chance de a média ficar abaixo de 3,2 pela normal e pela simulação.

In [ ]:
notas = np.array([1, 2, 3, 4, 5])
probs = np.array([0.25, 0.05, 0.05, 0.15, 0.50])

mu = (notas * probs).sum()                                     # E(X)
sigma = np.sqrt(((notas - mu)**2 * probs).sum())               # desvio padrão
n = 40
erro_padrao = sigma / np.sqrt(n)
print(f"μ = {mu:.2f}   σ = {sigma:.3f}   σ/√n = {erro_padrao:.3f}")

medias = rng.choice(notas, size=(100_000, n), p=probs).mean(axis=1)   # 100.000 médias de 40 notas

print(f"P(x̄ < 3,2)  pela normal (TLC): {norm.cdf((3.2 - mu) / erro_padrao):.4f}")
print(f"P(x̄ < 3,2)  na simulação:      {(medias < 3.2).mean():.4f}")

fig, eixos = plt.subplots(1, 2, figsize=(12, 4))
eixos[0].bar(notas, probs, color="steelblue", edgecolor="black")
eixos[0].set_title("Notas individuais (formato de U)")
eixos[0].set_xlabel("Estrelas")
x = np.linspace(2.5, 4.6, 300)
eixos[1].hist(medias, bins=50, density=True, color="lightsteelblue", edgecolor="gray")
eixos[1].plot(x, norm.pdf(x, mu, erro_padrao), color="firebrick", linewidth=2, label="normal do TLC")
eixos[1].set_title("Médias de 40 notas (formato de sino)")
eixos[1].set_xlabel("Média das 40 notas")
eixos[1].legend()
plt.show()

**Como ler:**
- À esquerda, as notas individuais formam um "U", com a maior parte em 1 e em 5. À direita, as **médias de 40 notas** formam um **sino** centrado em 3,6, e a curva normal do TLC se encaixa no histograma.
- A chance de a média ficar abaixo de 3,2 é de cerca de **6,7%** pela normal, e a simulação dá um valor próximo (as médias de 40 notas inteiras só assumem alguns valores, por isso a pequena diferença).

**Conclusão para o time:** uma queda da média para menos de 3,2 acontece por acaso em cerca de 1 a cada 15 janelas de 40 avaliações. É pouco frequente, mas não é raro o bastante para concluir sozinho que o app piorou.

## 2. Um estimador viciado: a variância "com $n$"

### A situação e a pergunta
Para estimar a variância das notas, alguém do time usou a fórmula "natural"
$$\hat\sigma^2 = \frac{1}{n}\sum_{i=1}^{n} (X_i - \bar X)^2,$$
dividindo por $n$. Outra pessoa usou a **variância amostral** $S^2$, que divide por $n - 1$. **Qual das duas acerta, em média?**

### A ferramenta da aula
Um estimador é **não viciado** (não viesado) se $E(\hat\theta) = \theta$: em média, ele acerta o parâmetro. A aula demonstrou que
$$E(\hat\sigma^2) = \sigma^2\left(1 - \frac{1}{n}\right),$$
ou seja, $\hat\sigma^2$ **subestima** $\sigma^2$, enquanto $E(S^2) = \sigma^2$.

### A resposta pela conta
Com amostras de $n = 5$ notas: $E(\hat\sigma^2) = 2{,}84 \times \left(1 - \frac{1}{5}\right) = 2{,}84 \times 0{,}8 = 2{,}272$, enquanto $E(S^2) = 2{,}84$.

### O que o código faz
Sorteia **100.000 amostras de 5 notas** e calcula os dois estimadores em cada uma. No `numpy`, a opção `ddof` escolhe o divisor: `ddof=0` divide por $n$ e `ddof=1` divide por $n - 1$. Depois tira a **média** de cada estimador sobre as 100.000 amostras.

In [ ]:
amostras = rng.choice(notas, size=(100_000, 5), p=probs)       # 100.000 amostras de 5 notas

sigma2_chapeu = amostras.var(axis=1, ddof=0)                   # divide por n
s2 = amostras.var(axis=1, ddof=1)                              # divide por n - 1

print(f"Variância verdadeira σ² = {sigma**2:.3f}")
print(f"Média de σ̂² (divide por n):     {sigma2_chapeu.mean():.3f}   (fórmula: σ²(1 - 1/n) = {sigma**2 * 0.8:.3f})")
print(f"Média de S² (divide por n - 1): {s2.mean():.3f}   (fórmula: σ² = {sigma**2:.3f})")

**Como ler:** dividindo por $n$, a média das estimativas fica em cerca de **2,27**, abaixo do valor verdadeiro 2,84: o estimador erra **sistematicamente para baixo**, exatamente o vício de $-\sigma^2/n$ da demonstração da aula. Dividindo por $n - 1$, a média das estimativas acerta **2,84**.

É por isso que calculadoras, planilhas e programas de estatística usam $n - 1$ na variância amostral.

## 3. Consistência: mais dados, estimativa mais perto do parâmetro

### A situação e a pergunta
O time acompanha a nota média de um app **novo** à medida que as avaliações chegam. A média verdadeira (que o time não conhece) é $\mu = 3{,}6$. **O que acontece com a estimativa $\bar x$ conforme o número de avaliações cresce?**

### A ferramenta da aula
Um estimador é **consistente** quando, à medida que $n$ cresce, a chance de ele errar por mais que qualquer margem $\epsilon$ vai a zero: $\lim_{n\to\infty} P(|\hat\theta - \theta| > \epsilon) = 0$. Para a média, isso vem de $\text{Var}(\bar X) = \sigma^2/n \to 0$.

### O que o código faz
1. sorteia **5 sequências** de 2.000 avaliações (5 "apps" com a mesma distribuição de notas);
2. para cada sequência, calcula a média **acumulada** (a média das primeiras 1, 2, 3, ..., 2.000 avaliações);
3. desenha as 5 curvas e a linha do valor verdadeiro $\mu = 3{,}6$.

In [ ]:
for _ in range(5):
    sequencia = rng.choice(notas, size=2000, p=probs)          # 2.000 avaliações em sequência
    media_acumulada = np.cumsum(sequencia) / np.arange(1, 2001)   # média das primeiras k notas
    plt.plot(range(1, 2001), media_acumulada, linewidth=1)

plt.axhline(mu, color="black", linestyle="--", label="μ = 3,6")
plt.xscale("log")                                              # eixo em escala log, para ver o começo
plt.xlabel("Número de avaliações (escala logarítmica)")
plt.ylabel("Nota média até aquele momento")
plt.title("Consistência: a média amostral se aproxima de μ")
plt.legend()
plt.show()

**Como ler:** com poucas avaliações, as 5 curvas oscilam muito — um app pode parecer ter nota 2 ou nota 5 nas primeiras avaliações. Conforme as avaliações se acumulam, todas as curvas se **aproximam de 3,6** e as oscilações diminuem. (O eixo horizontal está em escala logarítmica: cada marca vale 10 vezes a anterior, para dar para ver o começo e o fim no mesmo gráfico.)

**Conclusão:** é por isso que lojas de aplicativos mostram "poucas avaliações" para apps novos: com $n$ pequeno, a média ainda não é confiável.

## 4. Eficiência: média ou mediana?

### A situação e a pergunta
Num serviço de streaming, o tempo de carregamento de um vídeo é normal com média $\mu = 3$ s e desvio padrão $\sigma = 1$ s. Para estimar $\mu$ com amostras de **25** vídeos, dá para usar a **média** ou a **mediana** da amostra: como a normal é simétrica, as duas acertam em média. **Qual das duas é melhor?**

### A ferramenta da aula
Entre dois estimadores **não viciados**, o mais **eficiente** é o de **menor variância**: ele erra menos de amostra para amostra.

### O que o código faz
Sorteia **100.000 amostras de 25 tempos**, calcula a média e a mediana de cada uma e compara a média e a variância dos dois estimadores.

In [ ]:
amostras = rng.normal(3, 1, size=(100_000, 25))               # 100.000 amostras de 25 tempos

est_media = amostras.mean(axis=1)
est_mediana = np.median(amostras, axis=1)

print(f"Média:    E = {est_media.mean():.3f}   Var = {est_media.var():.4f}   (fórmula: σ²/n = {1 / 25:.4f})")
print(f"Mediana:  E = {est_mediana.mean():.3f}   Var = {est_mediana.var():.4f}")
print(f"A variância da mediana é {est_mediana.var() / est_media.var():.2f} vezes a da média")

**Como ler:** os dois estimadores acertam em média (as duas médias ficam em 3,00), mas a **mediana varia cerca de 1,5 vez mais** que a média. Então, para dados normais, a média é **mais eficiente**: com o mesmo número de vídeos, ela dá estimativas mais precisas. Para a mediana ter a mesma precisão, seria preciso uma amostra cerca de 50% maior.

(A mediana tem outras vantagens: se a amostra tiver valores absurdos — um vídeo que travou e "demorou" 10 minutos —, ela é muito menos afetada que a média.)

## Para fechar

Neste notebook, o time de dados usou as ferramentas da Aula 16:

- o **TLC** transformou uma população em formato de "U" (as notas) numa média com formato de **sino**, e permitiu calcular probabilidades para a média de 40 avaliações;
- **ausência de vício:** dividir a variância por $n$ subestima $\sigma^2$; dividir por $n - 1$ ($S^2$) acerta em média;
- **consistência:** a média amostral se aproxima de $\mu$ conforme $n$ cresce;
- **eficiência:** para dados normais, a média é mais precisa que a mediana.

Na **Aula 18** vêm duas distribuições novas, a **t de Student** e a **qui-quadrado**, que aparecem quando precisamos **estimar $\sigma$** a partir da amostra.